# Sentiment — unigrams vs bigrams

Daily notebook — small experiment, fully reproducible (seed pinned below).

Does adding bigrams help a logistic regression on short sentiment snippets?

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.model_selection import learning_curve, validation_curve
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              GradientBoostingClassifier, GradientBoostingRegressor,
                              HistGradientBoostingClassifier, VotingClassifier,
                              IsolationForest)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, roc_auc_score, confusion_matrix,
                             classification_report, mean_squared_error,
                             mean_absolute_error, r2_score, silhouette_score)


## Data

In [ ]:
SEED = 1354

rng = np.random.RandomState(SEED)
pos = ["great movie, loved the acting", "fantastic plot and wonderful cast",
       "an excellent film, highly recommend", "brilliant direction, superb story"]
neg = ["terrible movie, waste of time", "awful plot and bad acting",
       "boring film, would not recommend", "poor direction, weak story"]
texts, labels = [], []
for _ in range(700):
    texts.append(rng.choice(pos) + " " + rng.choice(["really", "truly", "so", "very"]) + " good")
    labels.append(1)
for _ in range(700):
    texts.append(rng.choice(neg) + " " + rng.choice(["really", "truly", "so", "very"]) + " bad")
    labels.append(0)
df = pd.DataFrame({"review": texts, "positive": labels}).sample(frac=1, random_state=SEED).reset_index(drop=True)
flip = rng.rand(len(df)) < 0.04  # a little label noise
df.loc[flip, "positive"] = 1 - df.loc[flip, "positive"]
print("shape:", df.shape)
print(df.head(3).to_string())


shape: (1400, 2)
                                    review  positive
0      awful plot and bad acting truly bad         0
1  boring film, would not recommend so bad         0
2      awful plot and bad acting truly bad         0


## Comparison

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    df["review"], df["positive"], test_size=0.25, random_state=SEED, stratify=df["positive"])
for ng in [(1, 1), (1, 2)]:
    pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=ng)),
                     ("lr", LogisticRegression(max_iter=1000))])
    pipe.fit(X_train, y_train)
    acc = pipe.score(X_test, y_test)
    print(f"ngram {ng}: accuracy {acc:.4f} | vocab {len(pipe.named_steps['tfidf'].vocabulary_)}")


ngram (1, 1): accuracy 0.9429 | vocab 35
ngram (1, 2): accuracy 0.9429 | vocab 92


## Takeaways

- Bigrams help a little ('not good' patterns), but unigrams already carry most of the signal on this toy data.
- Vocab roughly doubles — worth it for the small gain here.